# Emotional analysis of GLQ histories

This notebook adds sentence-level emotion annotations to the previously clustered corpus. A llm assigns continuous scores for eight Plutchik emotion categories—anger, anticipation, disgust, fear, joy, sadness, surprise, and trust—plus a neutral score. The resulting annotations are used for exploratory sentence- and cluster-level analysis and are subsequently checked through a manually inspected stratified sample. The workflow is interpretive and exploratory: model scores are treated as analytical annotations rather than validated psychological measurements.


In [ ]:
import re
import numpy as np
import pandas as pd
from pprint import pprint

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

In [ ]:
df = pd.read_parquet('Writer_Embeddings_Clusters.parquet')


In [ ]:
# check
df.head(2)

# Emotion scoring

Each sentence is scored independently for the eight Plutchik emotion categories plus a neutral score. Scores range from 0 to 1 and multiple emotions may receive non-zero scores. These values are model-based analytical annotations, not human-coded ground truth.


In [ ]:
import os
import openai
from openai import OpenAI                         

In [ ]:
client = OpenAI()

In [ ]:
import json
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


In [ ]:
# Emotion labels
emotion_labels = ["neutral", "anger", "joy", "sadness", "fear", "trust", "disgust", "anticipation", "surprise"]

In [ ]:
# Prompt builder with validation
def gpt_emotion_batch(batch_df):
    batch_df = batch_df[
        batch_df["Sentences"].notna()
        & (batch_df["Sentences"].str.strip().str.len() > 5)
    ]

    if batch_df.empty:
        print("Skipping empty or invalid batch.")
        return {}

    batch_id_map = {
        str(idx): sentence
        for idx, sentence in zip(batch_df["ID"], batch_df["Sentences"])
    }

    numbered_sentences = "\n".join(
        [f"{i}. {s}" for i, s in batch_id_map.items()]
    )


    prompt = f"""
    You are analyzing affective expression in Greek historical discourse.

    For each sentence, score the intensity of the following eight emotions:
    anger, joy, sadness, fear, trust, disgust, anticipation, surprise.

    If you do NOT detect a clearly identifiable affective expression provide a neutral score.

    IMPORTANT:
    - Do not infer an emotion from evaluation alone. However, evaluative language may contribute to an emotion classification when its wording, rhetorical force, or context conveys a recognizable affective stance.
    - Assign an emotion only when the sentence conveys a reasonably identifiable
      affective orientation the writer is trying to convey.
    - Multiple emotions may be present in the same sentence.
    - Scores range from 0 (not present) to 1 (very intense).

    Trust refers to affective confidence, reliance, assurance, or faith.
    Do not use trust simply to represent approval, agreement, positive evaluation,
    certainty, or the presentation of an idea as correct.

    Return exactly these nine scores for every sentence: neutral, anger, joy, sadness, fear, trust, disgust, anticipation, surprise.

    Do not omit any category.

    Return the result as a JSON object where each key is the sentence number
    (as a string), and the value is another JSON object with emotion scores.

    Example format:
    {{
      "123": {{
    "neutral": 0.8,
    "anger": 0.0,
    "joy": 0.0,
    "sadness": 0.0,
    "fear": 0.0,
    "trust": 0.0,
    "disgust": 0.0,
    "anticipation": 0.1,
    "surprise": 0.0
      }}
    }}

    Sentences:
    {numbered_sentences}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You analyze affective expression in Greek historical discourse. "
                        "Distinguish emotion from positive or negative evaluation."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0,
            max_tokens=2000
        )

        reply = response.choices[0].message.content.strip()

        # Strip Markdown formatting if present
        if reply.startswith("```json"):
            reply = reply.replace("```json", "").strip()

        if reply.endswith("```"):
            reply = reply.replace("```", "").strip()

        # Validate response
        if not reply or not reply.startswith("{"):
            print(f"⚠️ Invalid or empty GPT response:\n{reply}")
            return {}

        # Parse JSON
        return json.loads(reply)

    except Exception as e:
        print(f"Error in batch: {e}")
        return {}


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

BATCH_SIZE = 5
MAX_WORKERS = 4

# Create batches
batches = [df.iloc[i:i + BATCH_SIZE] for i in range(0, len(df), BATCH_SIZE)]
results = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    future_to_batch = {executor.submit(gpt_emotion_batch, batch): batch for batch in batches}

    for future in tqdm(as_completed(future_to_batch), total=len(batches), desc="Processing batches"):
        output = future.result()
        results.update(output)
        time.sleep(1)  # Avoid hitting rate limits

In [ ]:
emotion_df = pd.DataFrame.from_dict(results, orient='index').astype(float)
emotion_df.index = emotion_df.index.astype(int)  # Ensure index matches corpus_df['ID']
emotion_df = emotion_df.sort_index()

In [ ]:
# Merge
corpus_df_emotions = df.set_index("ID").join(emotion_df, how='left').reset_index()

In [ ]:
emotion_cols = ["anger", "joy", "sadness", "fear", "trust", "disgust", "anticipation", "surprise"]

In [ ]:
corpus_df_emotions[emotion_cols].isna().any()

### Identify the dominant emotion per sentence

For descriptive inspection, the notebook records the highest-scoring non-neutral emotion for each sentence. The complete multilabel scores are retained.


In [ ]:
# Function to extract the top emotion and its score from each row
def extract_top_emotion_and_score(row, possible_emotions):
    # Initialize variables to track the top emotion and its score
    top_emotion = None
    top_score = -1  # Start with -1 to ensure any positive score will be higher
    
    # Iterate through all possible emotions to find the top one for this row
    for emotion in possible_emotions:
        # Check if this emotion's score is higher than the current top score
        emotion_score = row[emotion] if emotion in row else np.nan  # Adjusted to access the row directly
        if pd.notna(emotion_score) and emotion_score > top_score:
            top_emotion = emotion
            top_score = emotion_score
   
    return pd.Series([top_emotion, top_score], index=['Top Emotion', 'Top Emotion Score'])


In [ ]:
possible_emotions = ["anger", "joy", "sadness", "fear", "trust", "disgust", "anticipation", "surprise"]  # Adjust as needed

In [ ]:
# apply the function
corpus_df_emotions[['Top Emotion', 'Top Emotion Score']] = corpus_df_emotions.apply(lambda row: extract_top_emotion_and_score(row, emotion_cols), axis=1)
corpus_df_emotions.head()

### Save

In [ ]:
corpus_df_emotions.to_parquet("Writer_Clusters_Embeddings_Emotions.parquet", index = False) 

## Check

In [ ]:
# Find the highest-anger sentence within each semantic cluster
most_anger_sentences = (
    corpus_df_emotions.loc[corpus_df_emotions.groupby("Cluster_title2")["anger"].idxmax()]
    .sort_values(by="anger", ascending=False)
    .reset_index(drop=True)
)


In [ ]:
# Example
most_anger_sentences[["Sentences", "anger"]].head(20)

In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 100)

### Aggregate emotion profiles by semantic cluster

Mean emotion scores are calculated across the sentences assigned to each semantic cluster. These summaries describe the model-assigned affective profile of a cluster; they should not be interpreted as direct measurements of the historian’s psychological state.


In [ ]:
# Group by Topic_title and compute the mean for each emotion
emotions_columns = ["anger", "joy", "sadness", "fear", "trust", "disgust", "anticipation", "surprise"]

In [ ]:
emotion_summary = corpus_df_emotions.groupby('Cluster_title2')[emotions_columns].mean().reset_index()

In [ ]:
emotion_summary

In [ ]:
# Sort the df by a specific emotion column in descending order
most_angry_clusters = emotion_summary.sort_values(by="anger", ascending=False)
most_joyful_clusters = emotion_summary.sort_values(by="joy", ascending=False)
most_fearful_clusters = emotion_summary.sort_values(by="fear", ascending=False)
most_sad_clusters = emotion_summary.sort_values(by="sadness", ascending=False)
most_disgust_clusters = emotion_summary.sort_values(by="disgust", ascending=False)
most_surprised_clusters = emotion_summary.sort_values(by="surprise", ascending=False)
most_trust_clusters = emotion_summary.sort_values(by="trust", ascending=False)
most_anticipation_clusters = emotion_summary.sort_values(by="anticipation", ascending=False)


In [ ]:
# Display the top clusters for each emotion
print("Most Angry Clusters:")
most_angry_clusters[["Cluster_title2", "anger"]].head(5)

In [ ]:
print("\nMost Joyful Clusters:")
most_joyful_clusters[["Cluster_title2", "joy"]].head(5)

In [ ]:
print("\nMost Fearful Clusters:")
most_fearful_clusters[["Cluster_title2", "fear"]].head(5)

In [ ]:
print("\nMost Sad Clusters:")
most_sad_clusters[["Cluster_title2", "sadness"]].head(5)

In [ ]:
print("\nMost Disgust Clusters:")
most_disgust_clusters[["Cluster_title2", "disgust"]].head(5)

In [ ]:
print("\nMost Trusting Clusters:")
most_trust_clusters[["Cluster_title2", "trust"]].head(5)

# Qualitative validation sample

A stratified sample is drawn for manual inspection of the model-assigned emotion scores. 

In [ ]:
import re
import numpy as np
import pandas as pd
from pprint import pprint

In [ ]:
corpus_df_emotions.head(2)

In [ ]:
corpus_df_emotions.columns

In [ ]:
corpus_df_emotions.columns.tolist()

In [ ]:
corpus_df_emotions["neutral"].describe()

In [ ]:
corpus_df_emotions[[
    "Sentences",
    "neutral",
    "anger",
    "joy",
    "sadness",
    "fear",
    "trust",
    "disgust",
    "anticipation",
    "surprise",
    "Top Emotion",
    "Top Emotion Score"
]].tail(10)

In [ ]:
emotion_cols_with_neutral = [
    "neutral",
    "anger",
    "joy",
    "sadness",
    "fear",
    "trust",
    "disgust",
    "anticipation",
    "surprise"
]

corpus_df_emotions["Top_Category"] = corpus_df_emotions[emotion_cols_with_neutral].idxmax(axis=1)

corpus_df_emotions["Top_Category_Score"] = corpus_df_emotions[emotion_cols_with_neutral].max(axis=1)

In [ ]:
corpus_df_emotions["Top_Category"].value_counts()

In [ ]:
corpus_df_emotions.loc[
    corpus_df_emotions["Top_Category"] == "neutral",
    "Top_Category_Score"
].describe()

In [ ]:
corpus_df_emotions.columns

In [ ]:
print("Total sentences:", len(corpus_df_emotions))

print("\nTop Emotion distribution:")
print(corpus_df_emotions["Top_Category"].value_counts(dropna=False))

print("\nTop Emotion Score distribution:")
print(corpus_df_emotions["Top_Category_Score"].describe())

### Sampling bands for manual inspection

The low/medium/high cut-offs (0–0.2, >0.2–0.4, >0.4–1.0) are researcher-defined bins used to construct a varied validation sample. They are not intrinsic categories of the model.


In [ ]:
corpus_df_emotions["Intensity_Band"] = pd.cut(
    corpus_df_emotions["Top_Category_Score"],
    bins=[-0.01, 0.2, 0.4, 1.0],
    labels=["low", "medium", "high"]
)

pd.crosstab(
    corpus_df_emotions["Top_Category"],
    corpus_df_emotions["Intensity_Band"]
)

In [ ]:
sample_sizes = {    
    "neutral": 40,
    "anger": 25, 
    "trust": 20,
    "sadness": 15,
    "anticipation": 12,
    "joy": 12,
    "fear": 8,
    "disgust": 8,
    "surprise": 10}

sum(sample_sizes.values())

In [ ]:
samples = []

for category, n in sample_sizes.items():

    subset = corpus_df_emotions[
        corpus_df_emotions["Top_Category"] == category
    ]

    sample = subset.sample(
        n=n,
        random_state=42
    )

    samples.append(sample)

validation_sample = (
    pd.concat(samples)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

len(validation_sample)

In [ ]:
validation_sample["Top_Category"].value_counts()

In [ ]:
validation_sample.groupby("Top_Category")[
    "Top_Category_Score"
].agg(["count", "min", "median", "max"])

In [ ]:
scored_sample = validation_sample[
    ["ID", "Writer", "Chapters", "Sentences"]
    + emotion_cols_with_neutral
    + ["Top_Category", "Top_Category_Score", "Intensity_Band"]
].copy()


In [ ]:
pd.set_option('display.width', None)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', None) 
pd.set_option('display.max_columns', None)

In [ ]:
scored_sample

In [ ]:
sample_sizes = {
    "trust": 30,
    "anger": 30,
    "sadness": 20,
    "anticipation": 20,
    "joy": 15,
    "disgust": 12,
    "fear": 12,
    "surprise": 11
}

samples = []


In [ ]:
for emotion, total_n in sample_sizes.items():

    subset = corpus_df_emotions[corpus_df_emotions["Top Emotion"] == emotion]

    available_bands = [
        band for band in ["low", "medium", "high"]
        if len(subset[subset["Intensity_Band"] == band]) > 0
    ]

    base_n = total_n // len(available_bands)
    remainder = total_n % len(available_bands)

    for i, band in enumerate(available_bands):

        band_subset = subset[
            subset["Intensity_Band"] == band
        ]

        n = base_n + (1 if i < remainder else 0)
        n = min(n, len(band_subset))

        samples.append(
            band_subset.sample(
                n=n,
                random_state=42
            )
        )

validation_sample = (
    pd.concat(samples)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
)

In [ ]:
print("N =", len(validation_sample))

pd.crosstab(
    validation_sample["Top Emotion"],
    validation_sample["Intensity_Band"]
)

In [ ]:
annotation_sheet = validation_sample[
    ["ID", "Writer", "Chapters", "Sentences"]
].copy()

annotation_sheet["anger"] = ""
annotation_sheet["anticipation"] = ""
annotation_sheet["disgust"] = ""
annotation_sheet["fear"] = ""
annotation_sheet["joy"] = ""
annotation_sheet["sadness"] = ""
annotation_sheet["surprise"] = ""
annotation_sheet["trust"] = ""

annotation_sheet.head()

In [ ]:
emotion_cols = [
    "anger",
    "anticipation",
    "disgust",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "trust"
]

scored_sample = validation_sample[
    ["ID", "Writer", "Chapters", "Sentences"]
    + emotion_cols
    + ["Top Emotion", "Top Emotion Score", "Intensity_Band"]
].copy()


In [ ]:
pd.set_option('display.width', None)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', None) # Display full content of each cell without truncation

In [ ]:
scored_sample.tail(10)

## Cluster-level emotion visualisations

Neutral is excluded from these visual summaries so that differences among the eight emotion categories are easier to inspect. The figures therefore represent relative emotional profiles rather than overall emotionality.


In [ ]:
corpus_df_emotions['Cluster_title2'].unique()

In [ ]:
pd.set_option('display.width', None)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', None)

In [ ]:
corpus_df_emotions.columns

In [ ]:
pd.set_option('display.width', 50)        # Automatically adjust the width to display all content
pd.set_option('display.max_colwidth', 400)

In [ ]:
import matplotlib.pyplot as plt
import textwrap

In [ ]:
# Group by topic and calculate the mean for each emotion. I am excluding neutral...
emotion_means = corpus_df_emotions.groupby('Cluster_title2')[['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'anticipation']].mean()
emotion_means

### Exploratory vizualization 

In [ ]:
# Compute total emotionality score per cluster (excluding 'neutral')
emotion_cols = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust', 'anticipation']
emotion_means['emotionality'] = emotion_means[emotion_cols].sum(axis=1)
emotion_means


In [ ]:
# Sort and keep top 10 clusters
top10 = emotion_means.sort_values('emotionality', ascending=False).head(10)
top10_emotions = top10[emotion_cols]
top10_emotions

In [ ]:
top10_emotions_sorted = top10_emotions.loc[top10_emotions.sum(axis=1).sort_values(ascending=False).index]
top10_emotions_sorted

In [ ]:
# Wrap long cluster titles
wrapped_labels = ['\n'.join(textwrap.wrap(label, 40)) for label in top10_emotions_sorted.index]

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
top10_emotions_sorted.plot(kind='barh', stacked=True, ax=ax, colormap='tab10')

# Apply the wrapped labels and fix tick positions
ax.set_yticks(range(len(wrapped_labels)))
ax.set_yticklabels(wrapped_labels)

# Invert y-axis to have most emotional cluster on top
ax.invert_yaxis()

# Beautify
plt.title('Top 10 Most Emotional Clusters (Writer)', fontsize=14)
plt.ylabel('Cluster')
plt.xlabel('Mean Emotional Score')
plt.xticks(rotation=45)

# Legend
plt.legend(title='Emotions', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Show
plt.savefig("Mean_emotional_scores_top10_clusters_Writer.png", bbox_inches='tight')

plt.show()

In [ ]:
# Wrap long cluster titles to improve visibility
wrapped_labels = ['\n'.join(textwrap.wrap(label, 45)) for label in top10_emotions.index]

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
top10_emotions.plot(kind='barh', stacked=True, ax=ax, colormap='tab10')

# Apply the wrapped labels
ax.set_yticklabels(wrapped_labels)

# Beautify
plt.title('Top 10 Most Emotional Clusters (Writer)', fontsize=14)
plt.ylabel('Cluster')
plt.xlabel('Mean Emotional Score')
plt.xticks(rotation=45)

# Legend
plt.legend(title='Emotions', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.savefig("Mean_emotional_scores_top10_clusters_Writer.png", bbox_inches='tight')

plt.show()